# Ray Train Barrier MVP

This notebook demonstrates a minimal prototype for testing the barrier method in Ray Train. Blocking `sleep` calls are used to emulate a deep learning prediction loop, and Ray's built-in barrier is used to synchronize distributed workers.

## 1. Import Required Libraries

Import Ray, time, and other required libraries for distributed execution and synchronization.

In [1]:
import ray
from ray import train
from ray.train.torch import TorchTrainer
from ray.train import get_context
from ray.train.collective import barrier
import time
import random
import os

## 2. Initialize Ray and Define Barrier Method

Initialize Ray and describe the use of Ray Train's built-in barrier for worker synchronization.

In [2]:
# Ensure Ray is fully shut down before setting environment variables
if not ray.is_initialized():
    ray.init()

print(f"Ray initialized: {ray.is_initialized()}")

# Ray Train's barrier will be used in the worker function below.

2026-04-30 17:09:21,777	INFO worker.py:2012 -- Started a local Ray instance.


Ray initialized: True


/Users/jackson/Research/code/histotools/PatchSorter/.venv/lib/python3.13/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## 3. Simulate Deep Learning Prediction Loop with Sleep

Define a function that simulates a deep learning prediction loop by sleeping for a random or fixed duration to emulate computation. The function will run M pred batches per cycle, synchronizing with a barrier after each cycle.

In [3]:
def simulated_pred_loop(config):
    import time
    import random
    import numpy as np
    import logging
    import os
    from ray.train import get_context
    from ray.train.collective import barrier

    n_cycles = config.get("n_cycles", 2)
    sleep_mean = config.get("sleep_mean", 5.0)
    sleep_std = config.get("sleep_std", 0.5)

    context = get_context()
    rank = context.get_world_rank()
    results = []

    # Set up a logger for each worker, writing to a file in the prototyping directory
    base_log_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    log_filename = os.path.join(base_log_dir, f"worker_{rank}.log")
    logger = logging.getLogger(f"worker_{rank}")
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(log_filename)
    fh.setLevel(logging.INFO)
    formatter = logging.Formatter('%(asctime)s %(levelname)s %(message)s')
    fh.setFormatter(formatter)
    if logger.hasHandlers():
        logger.handlers.clear()
    logger.addHandler(fh)

    logger.info(f"[Worker {rank}] Starting loop with {n_cycles} cycles at {time.time():.2f}.")

    for cycle in range(n_cycles):
        cycle_info = {"cycle": cycle, "sleep_time": None, "rank": rank, "barrier_wait": None, "events": []}
        # Wait a random time per cycle (normal distribution, mean=5, std=0.5)
        sleep_time = float(np.clip(np.random.normal(sleep_mean, sleep_std), 0, None))
        t0 = time.time()
        logger.info(f"[Worker {rank}] Cycle {cycle}: sleeping for {sleep_time:.2f} seconds at {t0:.2f}.")
        cycle_info["events"].append(("sleep_start", t0))
        time.sleep(sleep_time)
        t1 = time.time()
        elapsed = t1 - t0
        logger.info(f"[Worker {rank}] Cycle {cycle}: finished sleeping in {elapsed:.2f} seconds at {t1:.2f}.")
        cycle_info["events"].append(("sleep_end", t1))
        cycle_info["sleep_time"] = elapsed
        # Barrier sync
        t_barrier_start = time.time()
        logger.info(f"[Worker {rank}] Cycle {cycle}: waiting at barrier at {t_barrier_start:.2f}.")
        cycle_info["events"].append(("barrier_start", t_barrier_start))
        barrier()
        t_barrier_end = time.time()
        logger.info(f"[Worker {rank}] Cycle {cycle}: passed barrier at {t_barrier_end:.2f}.")
        cycle_info["events"].append(("barrier_end", t_barrier_end))
        barrier_wait = t_barrier_end - t_barrier_start
        cycle_info["barrier_wait"] = barrier_wait
        results.append(cycle_info)
    logger.info(f"[Worker {rank}] Finished all cycles at {time.time():.2f}.")
    return results


## 4. Launch Distributed Workers with Barrier Synchronization

Use Ray Train's TorchTrainer to launch multiple distributed workers, each running the simulated prediction loop and synchronizing at the barrier.

In [4]:
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig

trainer = TorchTrainer(
    train_loop_per_worker=simulated_pred_loop,
    train_loop_config={
        "n_cycles": 3,
        "sleep_mean": 5.0,
        "sleep_std": 0.5,
    },
    scaling_config=ScalingConfig(
        num_workers=3,  # Number of distributed workers
        use_gpu=False,
    ),
)

result = trainer.fit()

# Defensive: handle None result.metrics or missing 'result' key
results = None
if result is not None and hasattr(result, "metrics") and result.metrics is not None:
    results = result.metrics.get("result", None)

(TrainController pid=44162) Requesting resources: {'CPU': 1} * 3
(TrainController pid=44162) Attempting to start training worker group of size 3 with the following resources: [{'CPU': 1}] * 3
(RayTrainWorker pid=44169) Setting up process group for: env:// [rank=0, world_size=3]
(PlacementGroupCleaner pid=44165) Exception in thread PlacementGroupCleanerMonitor:
(PlacementGroupCleaner pid=44165) Traceback (most recent call last):
(PlacementGroupCleaner pid=44165)   File "/Users/jackson/.local/share/uv/python/cpython-3.13.11-macos-aarch64-none/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
(PlacementGroupCleaner pid=44165)     self.run()
(PlacementGroupCleaner pid=44165)     ~~~~~~~~^^
(PlacementGroupCleaner pid=44165)   File "/Users/jackson/.local/share/uv/python/cpython-3.13.11-macos-aarch64-none/lib/python3.13/threading.py", line 995, in run
(PlacementGroupCleaner pid=44165)     self._target(*self._args, **self._kwargs)
(PlacementGroupCleaner pid=44165)     ~~~~~~~~~~~~^^

In [5]:

# Defensive: handle None result.metrics or missing 'result' key
results = None
if result is not None and hasattr(result, "metrics") and result.metrics is not None:
    results = result.metrics.get("result", None)

## 5. Collect and Display Results

Gather the results from all workers and display timing or synchronization information to verify the barrier method.

In [6]:
import pprint

print("Raw Ray result object:")
print(result)
print("\nRay result.metrics:")
print(getattr(result, "metrics", None))

if results is not None:
    print("Barrier MVP Results (per worker):")
    pprint.pprint(results)
else:
    print("No results returned from workers.")

Raw Ray result object:
Result(metrics=None, checkpoint=None, error=None, path='/Users/jackson/ray_results/ray_train_run-2026-04-30_17-09-22', metrics_dataframe=None, best_checkpoints=[], _storage_filesystem=<pyarrow._fs.LocalFileSystem object at 0x1224b8df0>)

Ray result.metrics:
None
No results returned from workers.
